# 金融數據工作坊 — 機器學習

## 工作坊系列回顧

我們的工作坊已經走過三場：

- **[網路爬蟲](https://python-practical-workshop.github.io/crawler/)**：把資料「抓下來」 — `NVDA.csv`、`AAPL.csv`、`GOOGL.csv`
- **[資料管理](https://python-practical-workshop.github.io/sql_flask/)**：把資料「存好、查得到、分享出去」 — sqlite、SQL、Flask
- **[資料統計與視覺化](https://python-practical-workshop.github.io/statistics_vis/)**：把資料「分析、加工」 — pandas、matplotlib

這一場，我們要往前一步：**不只是看懂資料，而是讓模型從資料中「學出規律」，用來輔助判斷。**  
本場以 `scikit-learn` 與 `XGBoost` 為主軸，用真實美股資料建立機器學習模型。

## 本場學習地圖

1. **機器學習基本觀念** — 監督式 / 非監督式、迴歸 / 分類，先建立詞彙與地圖
2. **特徵工程** — 把股價原始資料變成模型看得懂的「特徵」
3. **經典模型實作** — 迴歸（Linear Regression）、分類（決策樹、kNN）
4. **XGBoost** — 分類模型的主流選擇

# 機器學習基本觀念

## 傳統程式 vs 機器學習

- **傳統程式**：人寫規則，電腦依規則處理輸入 → 得到輸出
  - 例：「如果今天收盤價比昨天高，就標記為漲」→ 規則是人想的
- **機器學習**：給電腦大量「輸入 + 正確答案」，讓它自己找出規則
  - 例：給模型過去 5 年的股價資料 + 「隔天是漲是跌」的正確答案，讓模型自己找出漲跌的規律

**機器學習 = 用歷史資料 (X, Y) 學出一個函數 f(X) ≈ Y，再拿新的 X 去預測未知的 Y**

## 監督式學習 vs 非監督式學習

| | 監督式學習 (Supervised) | 非監督式學習 (Unsupervised) |
|---|---|---|
| 有沒有「正確答案」 | 有（叫做 label / target） | 沒有 |
| 目標 | 學會用 X 預測 Y | 找出資料本身的結構、分組 |
| 股價資料的例子 | 用今天的技術指標，預測明天股價是漲是跌 | 把股票依照走勢特性自動分成幾類 |
| 本場對應章節 | 迴歸、分類、XGBoost | K-means 分群 |

### 監督式學習：迴歸 vs 分類

監督式學習依照「Y 是什麼型態」分成兩種：

| | 迴歸 (Regression) | 分類 (Classification) |
|---|---|---|
| Y 的型態 | 連續數值 | 離散類別 |
| 股價例子 | 預測明天收盤價是多少元 | 預測明天是漲（U）還是跌（D） |
| 常見評估指標 | MAE、MSE、R² | Accuracy、Confusion Matrix |
| 本場模型 | Linear Regression、XGBRegressor | 決策樹、kNN、SVM、XGBClassifier |

## 詞彙表

| 詞彙 | 意思 |
|---|---|
| **Feature（特徵，X）** | 拿來預測用的欄位，例如 5 日均線、RSI |
| **Label / Target（標籤，Y）** | 想預測的答案，例如明天收盤價、明天漲跌 |
| **Train（訓練）** | 用已知 (X, Y) 讓模型學規律 |
| **Test（測試）** | 用模型沒看過的資料，檢查它學得好不好 |
| **Predict（預測）** | 拿新的 X，讓模型輸出它猜的 Y |
| **Overfitting（過擬合）** | 模型把訓練資料「背」起來，遇到新資料表現卻很差 |

> 💡 之後每次看到程式碼裡的 `X`、`y`、`.fit()`、`.predict()`，都對應到這裡的詞彙。

# 準備資料與環境

前三場是先爬蟲、存成 csv，再讀進 pandas。  
這場改用上一場附錄提過的 `yfinance`，不用額外上傳檔案。


In [ ]:
!pip install yfinance -q

In [ ]:
import pandas as pd
import yfinance as yf

# 單檔股票
df = yf.Ticker("NVDA").history(period="5y")

# 多檔股票收盤價
tickers = ["NVDA", "AAPL", "GOOGL", "MSFT", "JPM", "WMT", "KO", "SBUX"]
price = yf.download(tickers, period="5y", progress=False)["Close"]

## 了解資料樣貌
新的資料來源，比照上一場「了解資料樣貌」的動作，先確認後再用。

### 動作 1：簡單看資料樣貌 — `head()` / `tail()`

**要關注的重點**
1. **資料的範圍**：時間從何時開始何時結束
2. **資料排序方式**：日期升序
3. **資料在此排序上的量級差異**

In [ ]:
df.head(3)

In [ ]:
df.tail(3)

In [ ]:
price.head(3)

In [ ]:
price.tail(3)

### 動作 2：看健康狀況 — `info()`

**要關注的重點**
1. **每欄 Non-Null 數字一樣嗎？** 都一樣，無缺值
2. **Dtype 合理嗎？** 價格資訊應該要是 float，成交量會是 int
3. **資料量會不會太大？** 5年資料 1255筆，資料量都是 KB 級，完全沒問題

In [ ]:
df.info()

In [ ]:
price.info()

### 動作 3：看分布 — `describe()`

**要關注的重點**
1. **min 跟 max 合理嗎？** 出現 0 或負數？出現天文數字？
2. **mean 跟 50%（中位數）差很多嗎？** 差很多代表偏態或有離群值
3. **std 跟 mean 的比例？** std 比 mean 大 → 變異很大

In [ ]:
df.describe()

In [ ]:
price.describe()

### 動作 4：看缺值 — `isnull().sum()`

In [ ]:
df.isnull().sum()

In [ ]:
price.isnull().sum()

**小結**：8 檔股票、5 年資料、沒有缺值、dtype 都是 float，可以放心拿去用。

## 繪圖環境設定

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-noto-cjk -qq

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font = FontProperties(fname=font_path, size=12)

# 特徵工程


上一場學過 `pct_change`、`groupby`等「資料加工」技巧。  
> [pandas 的統計方法](https://pandas.pydata.org/docs/reference/series.html#computations-descriptive-stats)

原始的 Open / High / Low / Close / Volume 本身資訊有限，  
所以實務上會加工出更有意義的技術指標，當成模型的**特徵（Feature）**：

> [股票特徵參考](https://www.investing.com/equities/nvidia-corp-technical)



In [ ]:
df = df.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
df["Date"] = df["Date"].dt.tz_localize(None)
df = df.sort_values("Date").reset_index(drop=True)
df.tail()

## 報酬率
- **DailyReturn**：每日報酬率，抓「漲跌幅度」
- **Volatility_5**：5 日報酬率的標準差，抓「波動程度」

In [ ]:
df["DailyReturn"] = df["Close"].pct_change()
df["Volatility_5"] = df["DailyReturn"].rolling(5).std()
df[["Date", "Close", "DailyReturn", "Volatility_5"]].tail()

## [MA](https://rich01.com/what-is-moving-average-line/)
MA 代表過去一段時間裡的平均成交價格，  
最主要目的是用來判斷趨勢，通常是預期市場現在跟未來可能的走勢。

In [ ]:
df["MA_5"] = df["Close"].rolling(5).mean()
df["MA_20"] = df["Close"].rolling(20).mean()
df[["Date", "Close", "MA_5", "MA_20"]].tail()

### [練習] 算出 10 日均線 `MA_10`


In [ ]:
df["MA_10"] = df["Close"].rolling(____).mean()
df[["Date", "Close", "MA_5", "MA_10", "MA_20"]].tail()

## [RSI](https://zh.wikipedia.org/zh-tw/%E7%9B%B8%E5%B0%8D%E5%BC%B7%E5%BC%B1%E6%8C%87%E6%95%B8)

RSI 用來衡量近期漲多還是跌多，數值介於 0~100：
- 漲的力道遠大於跌 → RSI 接近 70（超買）
- 跌的力道遠大於漲 → RSI 接近 30（超賣）

計算邏輯：把每天的漲跌拆成「漲幅」和「跌幅」兩欄，分別取 14 日平均，再算比值。

In [ ]:
delta = df["Close"].diff()
gain = delta.clip(lower=0)          # 只留漲的部分，跌的變 0
loss = -delta.clip(upper=0)         # 只留跌的部分，取正值

avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()

rs = avg_gain / avg_loss
df["RSI_14"] = 100 - (100 / (1 + rs))

df[["Date", "Close", "RSI_14"]].tail()

## 預測目標
- 「**明天**」的收盤價
- 「**明天**」的收盤價跟今天比是漲(U)還是跌(D)

⚠️ 常見誤區：使用到未來資料
- 特徵（MA_5、RSI_14 ...）用的是「**今天以前**」算得出來的資料
- Target（答案）要是「**明天**」的資料，用 `.shift(-1)` 把明天的值搬到今天這一列

In [ ]:
# 明天的收盤價
df["Target_Close"] = df["Close"].shift(-1)

# 明天漲(U)還是跌(D)
df["Target_UD"] = (df["Close"].shift(-1) > df["Close"]).mask(df["Close"].shift(-1).isna()).map({True: "U", False: "D"})

df[["Date", "Close", "Target_Close", "Target_UD"]].head()

In [ ]:
df[["Date", "Close", "Target_Close", "Target_UD"]].tail()

## 處理缺值：`dropna()`

不少加工在資料開頭或結尾一定會產生 `NaN`，  
例如 `MA_20` 前 19 天算不出來、最後一天沒有「明天」。  
建模前要先把這些列拿掉，不然模型會出錯。

In [ ]:
print("處理前：", df.shape)
df_model = df.dropna().reset_index(drop=True)
print("處理後：", df_model.shape)
df_model.head(3)

In [ ]:
df_model.tail(3)

# 監督式學習經典模型

## 切分訓練測試資料

> **⚠️ 常見誤區**：使用到未來資料訓練   

最常見的切分方式是**隨機**抽一部分當測試集，  
但股價是**時間序列資料**：如果隨機切分，測試集裡可能混入比訓練集更早的日期。   
  
這代表模型可能是「拿未來的資料去訓練，再回頭預測過去」，  
這在真實世界是不可能發生的，等於作弊，測出來的準確率沒有意義。  

**正確做法**：依照時間順序切，前面時間當訓練集、後面時間當測試集。
``` python
train_X = X.iloc[:-test_size]
test_X  = X.iloc[-test_size:]

train_y = y.iloc[:-test_size]
test_y  = y.iloc[-test_size:]
```

## 迴歸（預測明天收盤價）

| | 欄位 |
|---|---|
| **X（特徵）** | MA_5, MA_10, MA_20, RSI_14, DailyReturn, Volatility_5, Volume |
| **Y（標籤）** | Target_Close（明天的收盤價） |

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn import metrics

feature_cols = ["MA_5", "MA_10", "MA_20", "RSI_14", "DailyReturn", "Volatility_5", "Volume"]

X = df_model[feature_cols]
y = df_model["Target_Close"]

In [ ]:
test_size = 60  # 用最後 60 個交易日當測試集

train_X = X.iloc[:-test_size]
test_X  = X.iloc[-test_size:]

train_y = y.iloc[:-test_size]
test_y  = y.iloc[-test_size:]

print(f"訓練集：{len(train_X)} 筆　測試集：{len(test_X)} 筆")

### 模型：線性回歸

In [ ]:
reg = LinearRegression()
reg.fit(train_X, train_y)

pred_y = reg.predict(test_X)

#### 準確率判斷：
- **MAE（平均絕對誤差）**：平均每次預測跟實際差多少「元」，越接近 0 越好
- **MSE（均方誤差）**：跟 MAE 類似，但誤差大的會被放大更多（懲罰離群的錯誤預測）
- **R²**：模型能解釋多少比例的變化，1 代表完美預測，0 代表跟盲猜差不多，負數代表比盲猜還差

In [ ]:
print("MAE:", metrics.mean_absolute_error(test_y, pred_y))
print("MSE:", metrics.mean_squared_error(test_y, pred_y))
print("R²:", reg.score(test_X, test_y))

#### [練習] 只用 `MA_5` 和 `Volume` 兩個特徵重新訓練，比較 R Squared 有沒有變化

In [ ]:
feature_cols_v2 = ["____", "____"]

X2 = df_model[feature_cols_v2]
train_X2 = X2.iloc[:-test_size]
test_X2  = X2.iloc[-test_size:]

reg2 = LinearRegression()
reg2.___(train_X2, train_y)
pred_y2 = reg2.___(test_X2)

print("R²:", reg2.score(test_X2, test_y))

## 分類（預測明天漲跌）

這次 Y 換成 `Target_UD`（明天是漲 U 還是跌 D），X 維持一樣。

In [ ]:
y_cls = df_model["Target_UD"]

train_y_cls = y_cls.iloc[:-test_size]
test_y_cls  = y_cls.iloc[-test_size:]

train_y_cls.value_counts()

### 模型：決策樹
決策樹的邏輯很直覺：不斷用「某個特徵是不是大於某個值？」把資料切分，直到能分出類別。

In [ ]:
from sklearn import tree

clf_tree = tree.DecisionTreeClassifier(max_depth=3, random_state=42)
clf_tree.fit(train_X, train_y_cls)

pred_tree = clf_tree.predict(test_X)
acc_tree = metrics.accuracy_score(test_y_cls, pred_tree)
print("決策樹 Accuracy:", acc_tree)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
tree.plot_tree(clf_tree, feature_names=feature_cols, class_names=clf_tree.classes_, filled=True, fontsize=8, ax=ax)
plt.show()

### 模型：kNN

kNN 的邏輯：看新資料點最接近的 k 個已知資料點，多數是漲就猜漲、多數是跌就猜跌。

In [ ]:
from sklearn import neighbors

clf_knn = neighbors.KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(train_X, train_y_cls)

pred_knn = clf_knn.predict(test_X)
acc_knn = metrics.accuracy_score(test_y_cls, pred_knn)
print("kNN Accuracy:", acc_knn)

### 準確率判斷：
baseline：永遠猜「多數類別」，這樣有一定的命中率，  
如果模型的準確率沒有明顯贏過 baseline，也就代表模型不佳。

> 資料量小、特徵簡單時，複雜模型不一定比簡單模型好。  

In [ ]:
baseline_pred = train_y_cls.value_counts().idxmax()  # 訓練集裡出現比較多次的類別
baseline_acc = (test_y_cls == baseline_pred).mean()

print(f"永遠猜 '{baseline_pred}' 的 baseline accuracy：{baseline_acc:.3f}")
print(f"決策樹 accuracy：{acc_tree:.3f}")
print(f"kNN accuracy：{acc_knn:.3f}")

#### [練習] 把 `n_neighbors` 從 5 改成 15，看 kNN accuracy 有沒有變化

In [ ]:
clf_knn_v2 = neighbors.KNeighborsClassifier(n_neighbors=____)
clf_knn_v2.fit(train_X, train_y_cls)

pred_knn_v2 = clf_knn_v2.predict(test_X)
print("kNN(k=15) Accuracy:", metrics.accuracy_score(test_y_cls, ____))

#### [練習] 把決策樹的 `max_depth` 從 3 改成 6，比較 accuracy 有沒有變好

In [ ]:
clf_tree_v2 = tree.DecisionTreeClassifier(max_depth=____, random_state=42)
clf_tree_v2.____(train_X, train_y_cls)

pred_tree_v2 = clf_tree_v2.predict(test_X)
print("決策樹(depth=6) Accuracy:", metrics.accuracy_score(test_y_cls, ____))

### Overfitting（過擬合）vs Underfitting（欠擬合）

- **Overfitting**：模型太複雜，把訓練資料的細節甚至雜訊都背下來 → 訓練集準確率很高，測試集卻很差
- **Underfitting**：模型太簡單，連訓練資料的規律都學不好 → 訓練集、測試集準確率都不高

> 決策樹的 `max_depth`（樹的深度）的深度越深，模型越複雜。

In [ ]:
depths = range(1, 15)
train_accs, test_accs = [], []

for d in depths:
    clf_d = tree.DecisionTreeClassifier(max_depth=d, random_state=42)
    clf_d.fit(train_X, train_y_cls)
    train_accs.append(metrics.accuracy_score(train_y_cls, clf_d.predict(train_X)))
    test_accs.append(metrics.accuracy_score(test_y_cls, clf_d.predict(test_X)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, train_accs, marker="o", label="訓練集 Accuracy")
ax.plot(depths, test_accs, marker="o", label="測試集 Accuracy")
ax.set_xlabel("max_depth（樹的深度）", fontproperties=font)
ax.set_ylabel("Accuracy")
ax.set_title("樹深度 vs Overfitting", fontproperties=font)
ax.legend(prop=font)
plt.show()

**怎麼看這張圖**：
- 深度越深，訓練集準確率通常會一直往上（甚至逼近 100%）
- 但測試集準確率到某個深度後就不再進步，甚至開始往下掉
- 兩條線分得越開，代表 overfitting 越嚴重

### Confusion Matrix（混淆矩陣）
Accuracy 只給一個總分，Confusion Matrix 能看出模型是「哪一種」錯得多：  
例如是把跌的誤判成漲的比較多，還是反過來。

In [ ]:
cm = metrics.confusion_matrix(test_y_cls, pred_tree, labels=["U", "D"])
cm_df = pd.DataFrame(cm, index=["實際 U", "實際 D"], columns=["預測 U", "預測 D"])
cm_df

#### [練習] 印出 kNN（`pred_knn`）的 confusion matrix，跟決策樹比較誰的錯誤集中在哪一格

In [ ]:
cm_knn = metrics.confusion_matrix(test_y_cls, ____, labels=["U", "D"])
cm_knn_df = pd.DataFrame(cm_knn, index=["實際 U", "實際 D"], columns=["預測 U", "預測 D"])
cm_knn_df

# XGBoost

單一決策樹容易 overfitting、預測力有限。實務上更常見的做法是聚集多棵樹來判斷(集眾人之力)：

| 集成方式 | 概念 | 代表演算法 |
|---|---|---|
| **Bagging** | 平行訓練很多棵獨立的樹，再把結果平均/投票 | Random Forest |
| **Boosting** | 一棵一棵樹依序訓練，每棵樹專門去修正前一棵樹犯的錯 | **XGBoost**、LightGBM |

XGBoost（eXtreme Gradient Boosting）之所以成為主流：
- 預測表現通常優於單一決策樹
- 訓練速度快、支援平行運算
- 內建處理缺值、有 `feature_importances_` 方便解釋模型
- 在 Kaggle 競賽中長年名列前茅

In [ ]:
!pip install xgboost -q

## XGBoost vs 決策樹

In [ ]:
from xgboost import XGBClassifier

train_y_bin = (train_y_cls == "U").astype(int)
test_y_bin  = (test_y_cls == "U").astype(int)

clf_xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
clf_xgb.fit(train_X, train_y_bin)

pred_xgb = clf_xgb.predict(test_X)
acc_xgb = metrics.accuracy_score(test_y_bin, pred_xgb)

print(f"決策樹 accuracy：{acc_tree:.3f}")
print(f"XGBoost accuracy：{acc_xgb:.3f}")
print(f"baseline accuracy：{baseline_acc:.3f}")

## Feature Importance：哪個特徵最重要？

這是 XGBoost 很好用的一個特性：能告訴你「模型主要靠哪些特徵做判斷」。

In [ ]:
importances = pd.Series(clf_xgb.feature_importances_, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot.barh(ax=ax)
ax.set_title("XGBoost Feature Importance（分類模型）", fontproperties=font)
plt.tight_layout()
plt.show()

## 參數調整

| 參數 | 意思 | 影響 |
|---|---|---|
| `n_estimators` | 要疊幾棵樹 | 越多通常越準，但太多會 overfitting、也變慢 |
| `max_depth` | 每棵樹的深度 | 越深越複雜，容易 overfitting |
| `learning_rate` | 每棵樹修正錯誤的「步伐」大小 | 越小學得越穩，但需要更多棵樹（`n_estimators` 要跟著調高） |


### [練習] 參數調整
把 `n_estimators` 改成 300、`max_depth` 改成 5，看分類 accuracy 變化

In [ ]:
clf_xgb_v2 = XGBClassifier(n_estimators=____, max_depth=____, learning_rate=0.1, random_state=42)
clf_xgb_v2.fit(train_X, train_y_bin)

pred_xgb_v2 = clf_xgb_v2.predict(test_X)
print("XGBoost(n=300, depth=5) Accuracy:", metrics.accuracy_score(test_y_bin, ____))

# [情境練習]

換一檔股票（例如 AAPL 或 GOOGL），重複一次「特徵工程 → 切分 → 訓練 → 比較」的流程。

## 選定股票

In [ ]:
stock_id = "AAPL"  # 換成 AAPL 或 GOOGL 都可以

df_p = yf.Ticker(stock_id).history(period="5y")
df_p = df_p.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
df_p["Date"] = df_p["Date"].dt.tz_localize(None)
df_p = df_p.sort_values("Date").reset_index(drop=True)

## 特徵工程

In [ ]:
df_p["DailyReturn"] = df_p["Close"].pct_change()
df_p["Volatility_5"] = df_p["DailyReturn"].rolling(5).std()

df_p["MA_5"] = df_p["Close"].____(5).mean()
df_p["MA_20"] = df_p["Close"].____(20).mean()

delta = df_p["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
df_p["RSI_14"] = 100 - (100 / (1 + gain.rolling(14).mean() / loss.rolling(14).mean()))

df_p["Target_UD"] = (df_p["Close"].shift(___) > df_p["Close"]).mask(df["Close"].shift(-1).isna()).map({True: "U", False: "D"})

df_p_model = df_p.____()  # 去掉 NaN
df_p_model.info()

## 切分訓練測試資料

In [ ]:
feature_cols_p = ["MA_5", "MA_20", "RSI_14", "DailyReturn", "Volatility_5", "Volume"]

X_p = df_p_model[feature_cols_p]
y_p = df_p_model["____"]

test_size_p = 60
train_X_p, test_X_p = X_p.iloc[:-test_size_p], X_p.iloc[____:]
train_y_p, test_y_p = y_p.iloc[:-test_size_p], y_p.iloc[____:]

## 模型比較

In [ ]:
results = {}

clf_tree_p = tree.DecisionTreeClassifier(max_depth=3, random_state=42)
clf_tree_p.fit(train_X_p, train_y_p)
results["Decision Tree"] = metrics.accuracy_score(test_y_p, clf_tree_p.____(test_X_p))

clf_knn_p = neighbors.KNeighborsClassifier(n_neighbors=5)
clf_knn_p.fit(train_X_p, train_y_p)
results["kNN"] = metrics.accuracy_score(test_y_p, clf_knn_p.predict(test_X_p))

train_y_p_bin = (train_y_p == "U").astype(int)
test_y_p_bin  = (test_y_p == "U").astype(int)
clf_xgb_p = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
clf_xgb_p.____(train_X_p, train_y_p_bin)
results["XGBoost"] = metrics.accuracy_score(test_y_p_bin, clf_xgb_p.predict(test_X_p))

baseline_p = (test_y_p == train_y_p.value_counts().idxmax()).mean()
results["Baseline"] = baseline_p

pd.Series(results).sort_values(ascending=False)

**討論**：三個模型的準確率跟 baseline 比起來如何？換一檔股票結果會一樣嗎？  
如果時間允許，也可以試著調整 `test_size`、`max_depth`，觀察結果怎麼變化。

# 延伸方向

本場只用了「數字型」的技術指標（MA、RSI、報酬率⋯）當特徵，  
但實務上牽動股價波動的因子遠不只這些——財報內容、新聞事件、法說會逐字稿、社群輿情，  
往往才是真正影響市場情緒與預期的關鍵，而這些都是文字，不是現成的數字欄位。

想把文字資料也變成模型看得懂的特徵，會需要用到自然語言處理（NLP）的技術，例如：
- 情緒分析（Sentiment Analysis）：把一則新聞標記成正面／負面／中性 ，當成新的一欄特徵加進 X
- 文字向量化（Embedding）：把整篇新聞、財報段落轉成一組數字向量，捕捉比「正負面標籤」更細緻的語意訊息，再接進原本的模型

也就是說，機器學習的功夫沒有在這裡結束，而是「特徵工程」的戰場從純數字延伸到了文字，  
這也是為什麼現在資料科學／金融科技的工作，愈來愈需要同時懂統計建模、也懂怎麼跟 LLM/NLP 工具打交道。

# 結語
我們此系列工作坊走到的資料流程：

- **抓** — 從網站把原始資料弄到本機（網路爬蟲）
- **存** — 結構化儲存、用 SQL 查（資料管理）
- **分析** — 資料探索、加工、視覺化（資料統計與視覺化）
- **判斷** - 讓模型從資料中學出規律，輔助判斷 (本場)

在資料領域對應的角色

| 職位 | 工作核心 | 典型技能 |
|---|---|---|
| **資料工程師 (DE)** | 收集資料、把資料從各處搬到「需要進階應用」的地方；同時也可能負責讓分析結果能被系統應用（部署、API） | ETL pipeline、資料倉儲、Airflow、dbt、Spark、Flask |
| **資料分析師 (DA)** | 把資料變成可以做決策的洞察 | SQL、pandas、統計、視覺化、商業理解 |
| **資料科學家 / 資料探勘** | 用資料建模、預測、做產品智慧 | 統計、機器學習、特徵工程、實驗設計 |


# 延伸學習資源

- [scikit-learn 官方教學 Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [scikit-learn 演算法選擇地圖](https://scikit-learn.org/stable/machine_learning_map.html)
- [XGBoost 官方文件](https://xgboost.readthedocs.io/)
- [林軒田教授 機器學習基石（YouTube）](https://www.youtube.com/@hsuantien/courses)
- [李宏毅教授 機器學習/深度學習課程（YouTube）](https://www.youtube.com/@HungyiLeeNTU/courses)

## 附錄 A：決策樹能做數值預測嗎？
我們可以把葉節點從輸出多數類別改成輸出「訓練資料的平均值」，  
概念上也等同於預測數值

In [ ]:
from xgboost import XGBRegressor

reg_xgb = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
reg_xgb.fit(train_X, train_y)

pred_xgb_reg = reg_xgb.predict(test_X)

print("Linear Regression R²:", reg.score(test_X, test_y))
print("XGBoost R²:", reg_xgb.score(test_X, test_y))
print("XGBoost MAE:", metrics.mean_absolute_error(test_y, pred_xgb_reg))

### 成效

決策樹都是靠「切分區間、取區間內訓練資料的平均值」來預測，  
這代表：**不管特徵長什麼樣，預測值永遠不會超出訓練資料 Y 的範圍。**

股價剛好是最容易踩到這個地雷的資料：  
如果股票在測試期間創新高，樹模型注定預測不準——因為它沒看過那麼高的價格。

In [ ]:
print("訓練集 Close 範圍：", train_y.min(), "~", train_y.max())
print("測試集 Close 範圍：", test_y.min(), "~", test_y.max())
print("XGBoost 預測值範圍：", pred_xgb_reg.min(), "~", pred_xgb_reg.max())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
test_dates = df_model["Date"].iloc[-test_size:]
ax.plot(test_dates, test_y.values, label="實際收盤價", marker="o", markersize=3)
ax.plot(test_dates, pred_xgb_reg, label="XGBoost 預測", marker="o", markersize=3)
ax.plot(test_dates, pred_y, label="Linear Regression 預測", marker="o", markersize=3)
ax.axhline(train_y.max(), color="gray", linestyle="--", linewidth=1, label="訓練集最高價")
ax.set_title("測試期間：實際股價 vs 兩個模型的預測", fontproperties=font)
ax.legend(prop=font)
plt.show()
# 股價衝出訓練集的最高價（灰色虛線）之後，
# XGBoost 的預測線幾乎貼著灰色虛線走不上去，Linear Regression 則能跟著趨勢往上延伸。

> 一個模型在某項任務表現很好，不代表它在所有任務上都是最佳選擇：   
> **選模型要看資料特性跟預測情境，沒有哪個模型能打天下。**

## 附錄 B：更多監督式分類模型

### SVM（Support Vector Machine）

SVM 的概念是找一條「間隔最大」的分界線，把兩個類別盡量分得越開越好。

⚠️ SVM 對特徵的**量級**很敏感——這裡 Volume 是億級、RSI_14 是 0~100，量級差太多會讓 SVM 學不好（甚至跑得很慢）。決策樹、kNN、XGBoost 都不太受量級影響，但 SVM 用之前一定要先標準化（`StandardScaler`）。

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X)
test_X_scaled = scaler.transform(test_X)

clf_svm = SVC(kernel="linear")
clf_svm.fit(train_X_scaled, train_y_cls)

pred_svm = clf_svm.predict(test_X_scaled)
print("SVM Accuracy:", metrics.accuracy_score(test_y_cls, pred_svm))

**解讀**：標準化之後 SVM 才能正常運作。這也是為什麼它不像決策樹、XGBoost 那樣可以直接套用原始數字——這類「事前處理要求」的差異，也是選模型時要考慮的成本之一。

### Random Forest（隨機森林）
XGBoost 走的是 boosting 一棵一棵接力修正錯誤，   
Random Forest 的樹則是 bagging 平行訓練很多棵**互相獨立**的決策樹，  
每棵只看部分資料、部分特徵，最後用投票決定結果。  

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
clf_rf.fit(train_X, train_y_cls)

pred_rf = clf_rf.predict(test_X)
acc_rf = metrics.accuracy_score(test_y_cls, pred_rf)
print("Random Forest Accuracy:", acc_rf)

### 分類模型比較

In [ ]:
comparison = pd.Series({
    "Baseline": baseline_acc,
    "Decision Tree": acc_tree,
    "kNN": acc_knn,
    "SVM": metrics.accuracy_score(test_y_cls, pred_svm),
    "Random Forest": acc_rf,
    "XGBoost": acc_xgb,
}).sort_values(ascending=False)
comparison

> 💡 這張表沒有「正確答案」——不同資料、不同特徵、不同時間切分，排名都可能不一樣。重點不是背誦哪個模型最強，而是知道每個模型的特性，才能判斷選的模型合不合理。

## 附錄 C：非監督式模型

### K-means 分群
非監督式學習沒有「正確答案」，只是讓機器依照資料的相似程度自動分組。

每一列代表**一檔股票**，欄位是這檔股票長期的「平均日報酬率」與「日報酬率波動度」——用這兩個數字來衡量股票彼此的相似程度。

In [ ]:
returns = price.pct_change()

stock_features = pd.DataFrame({
    "AvgReturn": returns.mean(),
    "Volatility": returns.std(),
})
stock_features.sort_values("Volatility")

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
stock_features["Cluster"] = kmeans.fit_predict(stock_features[["AvgReturn", "Volatility"]])
stock_features.sort_values("Cluster")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stock_features["AvgReturn"], stock_features["Volatility"], c=stock_features["Cluster"], cmap="viridis", s=100)
for name, row in stock_features.iterrows():
    ax.annotate(name, (row["AvgReturn"], row["Volatility"]), fontsize=10, xytext=(6, 6), textcoords="offset points")
ax.set_title("依「平均報酬率、波動度」對股票分群", fontproperties=font)
ax.set_xlabel("平均日報酬率", fontproperties=font)
ax.set_ylabel("日報酬率波動度", fontproperties=font)
plt.show()

**解讀**：NVDA 自成一群——報酬率、波動度都遠高於其他股票。JPM、KO、WMT 波動度最低，是防禦型的一群。剩下的 AAPL、GOOGL、MSFT、SBUX 波動度居中，被分在同一群——但這群**不完全等於「科技股」**：SBUX（星巴克）論產業是餐飲消費類股，但論「報酬率、波動度」這兩個統計特性，反而比較像蘋果、Google，而不是像可口可樂、沃爾瑪。

⚠️ 如果分群結果跟你腦中的產業直覺對不上，不代表模型做錯了——**機器抓的是你給它的「數字特徵」裡的相似性，不是人類認定的產業分類**。想要分群結果貼近產業直覺，就要把「所屬產業」這種資訊也放進特徵，不能只靠統計量。